# 01 - NETTOYAGE DES CORPUS

Pipeline d'ensemble : **Texte brut -> Nettoyage -> Tokenisation -> Annotation -> TF-IDF / Embeddings / MMR**

Objectif : retirer l'en-tête et le pied Gutenberg pour ne garder que le corps de l'oeuvre. La logique de nettoyage vit dans `pipeline.cleaning` ; ce notebook se contente de l'appliquer aux fichiers `raw/` et de produire `clean/`.

**Stratégie** : on cherche plusieurs marqueurs typiques d'un en-tête Gutenberg dans les premiers caractères du fichier (lignes "Produced by", références gallica/gutenberg.org, etc.) et on coupe au plus tardif. Côté fin : marqueur explicite "*** END OF ***", puis "FIN" isolé sur sa ligne, puis les bandes décoratives.


## Setup

In [1]:
import sys
from pathlib import Path

# Se déplacer dans le dossier du projet (NE LANCER Q'UNE FOIS)
%cd ../
%ls

/home/gau/projets/nlp/projet_nlp_app
README.md                audit_corpus.py     pg_catalog.csv
app/                     diagnostic_mmr.png  pipeline/
audit_clean_border.py    diagnostic_mmr.py   requirements.txt
audit_clean_borders.txt  figures/            scripts/
audit_corpus.csv         models/
audit_corpus.png         notebooks/


In [2]:
from pipeline import storage
from pipeline.cleaning import nettoyer, couper_debut, couper_fin, couper_decorations
from pipeline.config import CLEAN_PARAMS, PREFIXES

## Lister les livres a nettoyer

In [3]:
livres_raw = storage.list_objects(PREFIXES["raw"], suffix=".txt")
print(f"{len(livres_raw)} livres originaux a nettoyer")
for o in livres_raw[:5]:
    print(" -", o["Key"])

26 livres originaux a nettoyer
 - raw/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt
 - raw/adventure/raspe_rudolf_erich/pg50398_aventures_de_baron_de_munchausen.txt
 - raw/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt
 - raw/biography/marmont_auguste_frederic_louis_viesse_de_duc_de_raguse/pg30013_memoires_du_marechal_marmont_duc_de_raguse_39.txt
 - raw/biography/rouquette_louis_frederic/pg70801_lepopee_blanche.txt


## Test sur un livre

On fait tourner sur un seul livre pour visualiser ce qui est coupé et à quoi ressemble le début/fin du résultat. On affiche les coupes étape par étape.


In [4]:
cle_test = livres_raw[2]["Key"]
texte = storage.get_text(cle_test)

n0 = len(texte)
apres_debut = couper_debut(texte)
n1 = len(apres_debut)
apres_fin = couper_fin(apres_debut)
n2 = len(apres_fin)
propre = couper_decorations(apres_fin)
n3 = len(propre)

print(f"Livre : {cle_test}\n")
print(f"Brut         : {n0:>8}")
print(f"  - Header   : -{n0 - n1:>6}")
print(f"  - Footer   : -{n1 - n2:>6}")
print(f"  - Decor.   : -{n2 - n3:>6}")
print(f"Final        : {n3:>8} ({100 * n3 / n0:.1f}%)")
print()
print("=" * 70)
print("DEBUT (300 premiers caracteres)")
print("=" * 70)
print(propre[:300])
print()
print("=" * 70)
print("FIN (300 derniers caracteres) -- repr() pour voir espaces et sauts de ligne")
print("=" * 70)
print(repr(propre[-300:]))


Livre : raw/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt

Brut         :   333367
  - Header   : -   247
  - Footer   : -  4499
  - Decor.   : -     0
Final        :   328621 (98.6%)

DEBUT (300 premiers caracteres)
SOUVENIRS D'UNE ACTRICE

PAR

Mme LOUISE FUSIL.

     «Les années, les heures ne sont pas des mesures de la durée de la
     vie; une longue vie est celle dans laquelle nous nous sentons
     vivre; c'est une vie composée de sensations fortes et rapides, où
     tous les sentiments conserve

FIN (300 derniers caracteres) -- repr() pour voir espaces et sauts de ligne
"d'un ange\r\n     Tombée à l'ombre et regrettée aux cieux;\r\n     D'un peu de vie, oh! que la mort te venge.\r\n\r\n     Fleur dérobée au front d'un séraphin;\r\n     Reprends, ton rang avec un saint mystère,\r\n     Et ce fil d'or dont nous pleurons la fin\r\n     Va l'attacher autre part qu'à la terre!\r\n\r\nFIN."


## Traiter tous les livres et uploader sous `clean/`

On garde la structure de cles d'origine, préfixée par `clean/`, ce qui permet de retrouver l'arborescence par genre/auteur.


In [5]:
for obj in livres_raw:
    cle = obj["Key"]
    texte = storage.get_text(cle)
    propre = nettoyer(texte)

    sous_cle = cle[len(PREFIXES["raw"]):]
    nouvelle_cle = PREFIXES["clean"] + sous_cle

    storage.put_text(
        nouvelle_cle, propre,
        metadata={"source": "clean", "original_key": cle},
    )

    coupe = len(texte) - len(propre)
    ratio = 100 * len(propre) / len(texte) if texte else 0
    print(f"{cle}: {len(texte)} -> {len(propre)} chars (-{coupe}, {ratio:.0f}% restants)")


raw/adventure/dumas_alexandre/pg17990_le_comte_de_monte_cristo_tome_ii.txt: 743032 -> 723075 chars (-19957, 97% restants)
raw/adventure/raspe_rudolf_erich/pg50398_aventures_de_baron_de_munchausen.txt: 165849 -> 160370 chars (-5479, 97% restants)
raw/biography/fusil_louise/pg26720_souvenirs_dune_actrice_23.txt: 333367 -> 327679 chars (-5688, 98% restants)
raw/biography/marmont_auguste_frederic_louis_viesse_de_duc_de_raguse/pg30013_memoires_du_marechal_marmont_duc_de_raguse_39.txt: 682465 -> 682310 chars (-155, 100% restants)
raw/biography/rouquette_louis_frederic/pg70801_lepopee_blanche.txt: 257595 -> 252635 chars (-4960, 98% restants)
raw/biography/savary_anne_jean_marie_rene_duc_de_rovigo/pg20895_memoires_du_duc_de_rovigo_pour_servir_a_lhistoire_de_lempereur_napoleon_tome_2.txt: 590166 -> 589677 chars (-489, 100% restants)
raw/biography/stendhal/pg30977_la_vie_de_rossini_tome_i.txt: 448775 -> 390507 chars (-58268, 87% restants)
raw/historical_fiction/dumas_alexandre/pg17989_le_comte_d